# 🎯 Notebook 3 — Modélisation complète multi-couches (V12)

> Lance les entraînements des deux couches existantes et produit un rapport unifié.
>
> Pour le détail par couche, voir :
> - `couche1_planete/modelisation_couche1.ipynb` (62 cibles)
> - `couche2_sang/modelisation_couche2.ipynb` (10 cibles)

## Méthodologie
- Datasets : V12 Couche 1 (8 400 × 741) + V8 honnête Couche 2 (8 400 × 590)
- Anti-leak strict par cible
- 2 stratégies : Global / Global+Cluster
- Modèle : XGBoost 500 estim., depth=6, lr=0.05
- Split : GroupShuffleSplit par pays (test = pays jamais vus)

## 1. Imports

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# Charge les résultats des couches déjà entraînées
c1_results = 'couche1_planete/reports/results.csv'
c2_results = 'couche2_sang/reports/results.csv'

if os.path.exists(c1_results):
    res_c1 = pd.read_csv(c1_results)
    print(f'✓ Couche 1 : {len(res_c1)} cibles')
else:
    print('✗ Couche 1 — lancer : python couche1_planete/train.py')
    res_c1 = pd.DataFrame()

if os.path.exists(c2_results):
    res_c2 = pd.read_csv(c2_results)
    print(f'✓ Couche 2 : {len(res_c2)} cibles')
else:
    print('✗ Couche 2 — lancer : python couche2_sang/train.py')
    res_c2 = pd.DataFrame()

## 2. Tableau combiné des résultats

In [ ]:
if not res_c1.empty:
    res_c1['Couche'] = '🌍 Planète'
if not res_c2.empty:
    res_c2['Couche'] = '🩸 Sang'

all_res = pd.concat([res_c1, res_c2], ignore_index=True)
all_res = all_res.sort_values('R² Global+Cluster', ascending=False)
print(f'Total cibles : {len(all_res)}\n')

# Stats par catégorie
print('Distribution des R² :')
print(f'  ⭐⭐⭐ R² ≥ 0.85 : {(all_res["R² Global+Cluster"] >= 0.85).sum()}')
print(f'  ⭐⭐  R² ≥ 0.70 : {(all_res["R² Global+Cluster"] >= 0.70).sum()}')
print(f'  ⭐   R² ≥ 0.50 : {(all_res["R² Global+Cluster"] >= 0.50).sum()}')
print(f'  🟡 0.20-0.50  : {((all_res["R² Global+Cluster"] >= 0.20) & (all_res["R² Global+Cluster"] < 0.50)).sum()}')
print(f'  ❌ < 0.20     : {(all_res["R² Global+Cluster"] < 0.20).sum()}')

## 3. Top 20 cibles toutes couches confondues

In [ ]:
top20 = all_res.head(20)[['Couche','Cible','Technique','R² Global+Cluster','MAE']]
print(top20.to_string(index=False))

## 4. Visualisation comparative par couche

In [ ]:
fig, ax = plt.subplots(figsize=(12, max(8, len(all_res)*0.18)))
sorted_res = all_res.sort_values('R² Global+Cluster')
colors_map = {'🌍 Planète': 'steelblue', '🩸 Sang': 'crimson'}
colors = [colors_map.get(c, 'gray') for c in sorted_res['Couche']]
ax.barh(sorted_res['Cible'], sorted_res['R² Global+Cluster'], color=colors, alpha=0.85)
ax.axvline(0.5, color='gray', ls='--', alpha=0.5)
ax.axvline(0.7, color='green', ls='--', alpha=0.5)
ax.axvline(0, color='black', lw=0.5)
ax.set_xlim(-0.5, 1.0)
ax.set_xlabel('R² (pays jamais vus)')
ax.set_title(f'Performance des {len(all_res)} cibles toutes couches', weight='bold', fontsize=14)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=v, label=k) for k,v in colors_map.items()], loc='lower right')
plt.tight_layout()
plt.show()

## 5. Distribution R² par sous-couche

In [ ]:
sublayer_stats = all_res.groupby(['Couche','Sous-couche'])['R² Global+Cluster'].agg(
    ['mean','median','count','min','max']).round(3)
print(sublayer_stats.to_string())

## 6. Bilan & prochaines étapes

```
📊 BILAN V12

Couche 1 — La Planète (62 cibles)
  ⭐⭐⭐ Émissions CH4 / N2O      ≥ 0.95
  ⭐⭐  Énergie (charbon/pétrole)  ≥ 0.85
  ⭐⭐  Production œufs            0.85
  ⭐⭐  % forêt                    0.88
  ⭐⭐  Rendement lait             0.83
  ⭐   Cultures spé top           0.43-0.59 (concombre, tomate, colza)
  ❌   Cultures niche             Mangue, Pois chiche, Cerise

Couche 2 — Le Sang (10 cibles)  [strict env→démo]
  ⭐⭐⭐ Natalité, Fécondité       ≥ 0.95
  ⭐⭐  Mortalité infantile        0.82
  ⭐⭐  Espérance de vie           0.75
  ⭐   Croissance démo            0.69
  ❌   Migration nette            0.08 (→ Couche 4)
```

### À venir
- **Couche 3** — Le Moteur (économie, logistique, commerce)
- **Couche 4** — Le Chaos (politique, gouvernance, conflits)